# Interactive Algorithm Comparison Dashboard

Click legend entries to toggle algorithms. Double-click to isolate one.

In [9]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

run = 'a_20260419_115014'
# run = None  # Set to None to auto-detect the latest CSV file in results/

csv_file = f'../results/{run}/summary_{run}.csv' if run else None

results_candidates = [Path('results'), Path('../results')]
results_dir = next((p for p in results_candidates if p.exists() and p.is_dir()), None)
if results_dir is None:
    searched = [str(p.resolve()) for p in results_candidates]
    raise FileNotFoundError(f"Results folder not found. Searched: {searched}")

if csv_file is None:
    csv_candidates = sorted(results_dir.rglob('*.csv'))
    if not csv_candidates:
        raise FileNotFoundError(f"No .csv files found inside {results_dir.resolve()}")
    csv_file = str(csv_candidates[0])

print(f"Using CSV file: {csv_file}")
df = pd.read_csv(csv_file)

required_columns = ['instance', 'algorithm', 'objective_mean', 'exec_time_mean']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

best_objectives_candidates = [
    Path('best_solutions') / 'best_objectives.csv',
    Path('../best_solutions') / 'best_objectives.csv',
]
best_objectives_path = next((p for p in best_objectives_candidates if p.exists()), None)
if best_objectives_path is None:
    searched = [str(p.resolve()) for p in best_objectives_candidates]
    raise FileNotFoundError(f"Best objectives file not found. Searched: {searched}")

best_df = pd.read_csv(best_objectives_path)
if 'instance' not in best_df.columns or 'best_objective' not in best_df.columns:
    raise ValueError("best_objectives.csv must contain columns: instance, best_objective")

if 'dataset' in df.columns and 'dataset' in best_df.columns:
    merge_columns = ['dataset', 'instance']
else:
    merge_columns = ['instance']

df = df.merge(best_df[merge_columns + ['best_objective']], on=merge_columns, how='left')
ordered_columns = [col for col in df.columns if col != 'best_objective'] + ['best_objective']
df = df[ordered_columns]


def add_legend_mass_toggle(fig: go.Figure, y_position: float = 1.18) -> None:
    trace_count = len(fig.data)
    fig.update_layout(
        updatemenus=[
            dict(
                type='buttons',
                direction='right',
                x=1.0,
                y=y_position,
                xanchor='right',
                yanchor='bottom',
                showactive=False,
                buttons=[
                    dict(
                        label='Selecionar todos',
                        method='update',
                        args=[{'visible': [True] * trace_count}]
                    ),
                    dict(
                        label='Remover todos',
                        method='update',
                        args=[{'visible': ['legendonly'] * trace_count}]
                    ),
                ],
            )
        ]
    )


df.head()

Using CSV file: ../results/a_20260419_115014/summary_a_20260419_115014.csv


,dataset,algorithm,instance,total_runs,feasible_runs,infeasible_runs,timed_out_runs,objective_mean,objective_median,objective_mode,...,aisles_variance,aisles_std_dev,exec_time_mean,exec_time_median,exec_time_mode,exec_time_min,exec_time_max,exec_time_variance,exec_time_std_dev,best_objective
0,a,af_mixed_order-asc_prune-multi,instance_0001.txt,1,1,0,0,12.000,12.000,12.000,...,0.0,0.0,0.000864,0.000864,0.000864,0.000864,0.000864,0.0,0.0,15.000
1,a,af_mixed_order-asc_prune-multi,instance_0002.txt,1,1,0,0,1.000,1.000,1.000,...,0.0,0.0,0.000105,0.000105,0.000105,0.000105,0.000105,0.0,0.0,2.000
2,a,af_mixed_order-asc_prune-multi,instance_0003.txt,1,1,0,0,8.500,8.500,8.500,...,0.0,0.0,0.001055,0.001055,0.001055,0.001055,0.001055,0.0,0.0,12.000
3,a,af_mixed_order-asc_prune-multi,instance_0004.txt,1,1,0,0,2.600,2.600,2.600,...,0.0,0.0,0.001413,0.001413,0.001413,0.001413,0.001413,0.0,0.0,3.500
4,a,af_mixed_order-asc_prune-multi,instance_0005.txt,1,1,0,0,171.375,171.375,171.375,...,0.0,0.0,0.129426,0.129426,0.129426,0.129426,0.129426,0.0,0.0,177.875


In [10]:
# Objective Mean as % of Best Objective

objective_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='objective_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

best_objective_by_instance = (
    df[['instance', 'best_objective']]
    .drop_duplicates(subset='instance')
    .set_index('instance')['best_objective']
)

objective_pct_plot = objective_plot.div(best_objective_by_instance, axis=0) * 100
objective_pct_plot = objective_pct_plot.replace([float('inf'), -float('inf')], 0).fillna(0)

fig = go.Figure()
for algo in objective_pct_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=objective_pct_plot.index,
        y=objective_pct_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.1f}}% of best<extra></extra>'
    ))

fig.add_hline(y=100, line_dash='dash', line_color='black', annotation_text='best (100%)')

fig.update_layout(
    barmode='group',
    title='Objective Mean by Instance and Algorithm (% of Best Objective)',
    xaxis_title='Instance',
    yaxis_title='% of best_objective',
    legend_title='Algorithm',
    height=700,
    margin=dict(t=150),
    xaxis_tickangle=-45,
)
add_legend_mass_toggle(fig)
fig.show()

In [11]:
# Execution Time Mean

exec_time_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='exec_time_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

fig = go.Figure()
for algo in exec_time_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=exec_time_plot.index,
        y=exec_time_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.4f}}s<extra></extra>'
    ))

fig.update_layout(
    barmode='group',
    title='Execution Time Mean by Instance and Algorithm',
    xaxis_title='Instance',
    yaxis_title='exec_time_mean (s)',
    legend_title='Algorithm',
    height=600,
    margin=dict(t=150),
    xaxis_tickangle=-45,
)
add_legend_mass_toggle(fig)
fig.show()

In [12]:
# Rankings

ranking_dataset_param = None  # e.g., 'a', 'b', 'x' or None for all datasets

ranking_base = df[['instance', 'algorithm', 'objective_mean', 'exec_time_mean']].copy()
if ranking_dataset_param is not None and 'dataset' in df.columns:
    ranking_base = df[df['dataset'] == ranking_dataset_param][
        ['instance', 'algorithm', 'objective_mean', 'exec_time_mean']
    ].copy()

if ranking_base.empty:
    raise ValueError(f'No rows found for dataset={ranking_dataset_param!r}')


def build_final_ranking(base_df: pd.DataFrame, metric_column: str, ascending: bool) -> pd.DataFrame:
    metric_df = base_df.dropna(subset=[metric_column]).copy()
    if metric_df.empty:
        raise ValueError(f'No non-null values found for metric {metric_column!r}.')

    metric_df['N'] = metric_df.groupby('instance')['algorithm'].transform('nunique')
    metric_df['rank_class'] = metric_df.groupby('instance')[metric_column].rank(
        method='min', ascending=ascending
    ).astype(int)
    metric_df['score'] = metric_df['N'] - metric_df['rank_class']

    final_ranking = (
        metric_df.groupby('algorithm', as_index=False)['score']
        .agg(total_score='sum', average_score='mean', instances_count='count')
        .sort_values(['total_score', 'average_score', 'algorithm'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    final_ranking['final_rank'] = final_ranking['total_score'].rank(
        method='min', ascending=False
    ).astype(int)

    return final_ranking[
        ['final_rank', 'algorithm', 'total_score', 'average_score', 'instances_count']
    ]


objective_ranking = build_final_ranking(
    ranking_base, metric_column='objective_mean', ascending=False
)
exec_time_ranking = build_final_ranking(
    ranking_base, metric_column='exec_time_mean', ascending=True
)

print(f'Dataset filter for ranking: {ranking_dataset_param!r}')
print('\nOBJECTIVE_MEAN - FINAL RANKING')
display(objective_ranking)
print('\nEXEC_TIME_MEAN - FINAL RANKING')
display(exec_time_ranking)

Dataset filter for ranking: None

OBJECTIVE_MEAN - FINAL RANKING


,final_rank,algorithm,total_score,average_score,instances_count
0,1,af_useful_order-desc_prune-multi,624,31.20,20
1,2,af_useful_order-null_prune-multi,571,28.55,20
2,3,af_useful_order-desc_prune-simple,566,28.30,20
3,4,af_useful_order-desc_prune-null,549,27.45,20
4,5,af_mixed_order-desc_prune-multi,548,27.40,20
5,6,af_variety_order-desc_prune-multi,541,27.05,20
6,7,af_useful_order-asc_prune-multi,539,26.95,20
7,8,af_useful_order-null_prune-simple,498,24.90,20
8,9,af_variety_order-null_prune-multi,483,24.15,20
9,10,af_variety_order-desc_prune-simple,480,24.00,20



EXEC_TIME_MEAN - FINAL RANKING


,final_rank,algorithm,total_score,average_score,instances_count
0,1,af_useful_order-desc_prune-null,614,30.70,20
1,2,af_useful_order-asc_prune-null,597,29.85,20
2,3,af_variety_order-asc_prune-null,583,29.15,20
3,4,af_mixed_order-desc_prune-null,578,28.90,20
4,5,af_variety_order-desc_prune-null,568,28.40,20
5,6,af_mixed_order-null_prune-null,530,26.50,20
6,7,af_variety_order-null_prune-null,527,26.35,20
7,8,af_mixed_order-asc_prune-null,526,26.30,20
8,9,af_useful_order-null_prune-null,504,25.20,20
9,10,af_units_order-asc_prune-null,479,23.95,20
